# Ballot Processing
**Company:** Atlassian (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Onsite Loop, Arrays, Hash Tables, Sorting · **Difficulty/Frequency:** Common (6/10)


## Concepts

**What this problem is really testing:**
- Hash-map tallying
- Two-pass processing — for when a *later* fact decides what an *earlier* pass needed to record
- Multi-key sort tuples, for breaking ties in a defined order

**Why each one shows up here:**
- The point tally (3/2/1 by ranking position) is a straightforward hash-map job.
- The tricky part is the "first to reach the winning score" tiebreak: you can't know *when* a candidate first hit their final score until you already know what that final score *is* — and you only know that after seeing every ballot.
- Needing "the end" before you can even describe "the middle" is the classic sign that you need a **two-pass** algorithm.

**The one idea to hold onto:** whenever you need "the first time X happened" but X itself is only knowable at the very end, split the work into two passes: pass 1 learns the final facts, pass 2 replays the data now that those facts are known.

---

### Quick primers — the building blocks used below

**What is a Hash Map?**
- A hash map (Python `dict`) stores key → value pairs by hashing the key to a slot, giving **O(1) average** insert/lookup/delete.
- Used here to tally each candidate's running point total, and their per-position vote counts, in a single pass.

**Two-pass processing.**
- Some computations have a chicken-and-egg problem: you need a global fact (here, each candidate's *final* score) before you can figure out a smaller fact that depends on it (the *first* ballot where the running score matches that final value).
- There's no clever way around this — you just do it in two passes:
  - **Pass 1:** compute the global facts.
  - **Pass 2:** replay the same data, now that those facts are already known.

**Multi-key sort tuples, for deterministic tie-breaking.**
- Python's `sorted()`/`.sort()` compares tuples position by position, left to right.
- Negating a numeric field flips it from ascending to descending, even while the rest of the tuple stays in normal ascending order.
- So `(-points, first_reached, name)` reads as: "highest points first; if tied, whoever reached it earliest; if still tied, alphabetical" — all in one sort call.


## Problem Statement

Each ballot lists up to 3 candidates in preference order: 1st choice = 3 points, 2nd = 2, 3rd = 1. Return all candidates sorted by total points, descending.

**Follow-up -- tie handling**, two strategies:
1. **First to reach** -- among tied candidates, whoever's running total hit their *final* score earliest (by ballot index) wins the tie.
2. **Positional** -- compare vote counts at position 0 (3-pt votes), then position 1, then position 2.

**Example**

```python
ballots = [["A", "B"], ["B", "A"], ["C"]]
# A: 3 (ballot 0) + 2 (ballot 1) = 5
# B: 2 (ballot 0) + 3 (ballot 1) = 5
# C: 3 (ballot 2) = 3
# A and B are tied at 5 -- "first to reach 5": A hits 5 on ballot 1 (3+2), B hits 5 also on ballot 1 (2+3) -- a genuine tie in this constructed example, broken alphabetically.
```


### Approach 1 -- Single pass, points only (no tiebreak)

**Idea:** the foundation -- one pass over all ballots, adding `3 - pos` points per vote. This alone can rank candidates by score but has no way to break ties deterministically (relies on whatever order `dict.items()` happens to preserve, which is really "order first seen", not a stated rule).

**Time complexity:** O(V) where V is the total number of votes across all ballots.

**Space complexity:** O(m) for m distinct candidates.


In [ ]:
from typing import List, Dict


def tally_points(ballots: List[List[str]]) -> Dict[str, int]:
    """One pass: candidate -> total points. No tiebreak information collected."""
    points: Dict[str, int] = {}
    for ballot in ballots:
        for pos, candidate in enumerate(ballot):
            points[candidate] = points.get(candidate, 0) + (3 - pos)   # pos 0->3, 1->2, 2->1
    return points


### Approach 2 -- Optimal (two-pass, both tiebreak strategies)

**Idea:** Pass 1 tallies `points[c]` and, in the same sweep, `positional[c] = [count_at_0, count_at_1, count_at_2]` (free -- no extra pass needed for the *positional* strategy). Pass 2 replays the ballots with a running total per candidate and records `first_reached[c]` the instant that running total equals the final `points[c]` -- this is only meaningful because pass 1 already fixed what "final" means. Sort with `(-points[c], first_reached[c], c)` or `(-points[c], -positional[c][0], -positional[c][1], -positional[c][2], c)` depending on the requested strategy.

**Time complexity:** O(V) for each pass (two passes = O(V), constants aside) plus O(m log m) to sort m candidates.

**Space complexity:** O(m) for `points`, `positional`, and `first_reached`.


In [ ]:
def process_ballots(ballots: List[List[str]], tie_strategy: str = "first") -> List[str]:
    if not ballots:
        return []

    # Pass 1: tally points AND per-position vote counts in one sweep.
    points: Dict[str, int] = {}
    positional: Dict[str, List[int]] = {}
    for ballot in ballots:
        for pos, candidate in enumerate(ballot):
            if candidate not in points:
                points[candidate] = 0
                positional[candidate] = [0, 0, 0]
            points[candidate] += 3 - pos
            positional[candidate][pos] += 1

    # Pass 2 (only needed for "first"): replay, recording the earliest ballot index
    # where each candidate's RUNNING total first equals their FINAL total.
    first_reached: Dict[str, int] = {}
    if tie_strategy == "first":
        running = {c: 0 for c in points}
        for ballot_idx, ballot in enumerate(ballots):
            for pos, candidate in enumerate(ballot):
                running[candidate] += 3 - pos
                if running[candidate] == points[candidate] and candidate not in first_reached:
                    first_reached[candidate] = ballot_idx   # first time, never overwritten

    if tie_strategy == "positional":
        key = lambda c: (-points[c], -positional[c][0], -positional[c][1], -positional[c][2], c)
    else:
        key = lambda c: (-points[c], first_reached[c], c)

    return sorted(points.keys(), key=key)


## Verification

Check against the worked trace, both tie strategies, and the edge cases the Talking Points call out.

In [ ]:
ballots = [["A", "B"], ["B", "A"], ["C"]]
# A: ballot0 pos0(+3), ballot1 pos1(+2) -> running 3, then 5. final=5.
# B: ballot0 pos1(+2), ballot1 pos0(+3) -> running 2, then 5. final=5.
# C: ballot2 pos0(+3) -> final=3.
points = tally_points(ballots)
assert points == {"A": 5, "B": 5, "C": 3}

# "first" strategy: A and B both first hit their final score (5) on ballot index 1 -- a genuine
# tie even after the tiebreak, so alphabetical order (the final tuple element) decides.
result_first = process_ballots(ballots, "first")
assert result_first == ["A", "B", "C"]

# "positional": A has 1 vote at pos0 (ballots: pos0 once, pos1 once) vs B (pos0 once, pos1 once)
# -- identical positional profiles too -- so this also falls through to alphabetical.
result_pos = process_ballots(ballots, "positional")
assert result_pos == ["A", "B", "C"]

# A case where "first" and "positional" would clearly differ from naive insertion order:
ballots2 = [["X"], ["Y", "X"], ["X", "Y"]]
# X: 3 (b0) + 2 (b1) + 3 (b2) = 8   Y: 2 (b1) + 1 (b2) = 3 -- no real tie here, sanity check only
assert process_ballots(ballots2, "first") == ["X", "Y"]

# A genuine, resolvable "first-to-reach" tie: C reaches 4 on ballot 0, D reaches 4 on ballot 1.
ballots3 = [["C", "D"], ["D", "C"]]
# C: b0 pos0 -> 3, b1 pos1 -> +2 = 5   D: b0 pos1 -> 2, b1 pos0 -> +3 = 5
# Both hit 5 only at the very end (ballot 1) -- still tied; use a case with different finish points instead.
ballots4 = [["C"], ["D", "C"], ["D"]]
# C: b0 pos0(+3)=3, b1 pos1(+2)=5 -- reaches final(5) at ballot 1
# D: b1 pos0(+3)=3, b2 pos0(+3)=6 -- reaches final(6) at ballot 2
p4 = tally_points(ballots4)
assert p4 == {"C": 5, "D": 6}   # no tie here either -- points alone already orders them
assert process_ballots(ballots4, "first") == ["D", "C"]   # D has more points, wins outright

# Edge cases
assert process_ballots([], "first") == []
assert tally_points([]) == {}
single = process_ballots([["Solo"]], "first")
assert single == ["Solo"]

print("All checks passed.")


## Discussion -- remaining follow-up directions

- **Ballots with more than 3 candidates.** Generalize the point value to `len(ballot) - pos` (so the last-ranked candidate on an L-candidate ballot still gets 1 point), and size each candidate's `positional` list to `len(ballot)` (or track it as a `dict[pos] -> count]` if ballot lengths vary).
- **Return only the top K.** `return sorted(...)[:k]` after the existing sort -- no algorithmic change, just slicing the already-correct full ranking.
- **Streaming ballots (output ranking after each new ballot).** The "positional" strategy updates incrementally for free (just add to running counts). The "first" strategy is harder: it fundamentally needs to know the *final* score, which isn't stable until the stream ends -- so a truly streaming "first to reach" ranking would need to be provisional and potentially revised as more ballots arrive, or you accept O(m) re-derivation of `first_reached` after each new ballot (simpler, slower).
- **Candidate aliases (e.g. "Bob" and "Robert" are the same person).** Build a name -> canonical-name map (a small union-find if aliases can chain), and apply it when tallying so both spellings accumulate into one entry.
- **Duplicate candidates within one ballot.** Deduplicate per ballot before tallying (keep only the first occurrence of a repeated name, since it's already the higher-point mention) -- shown below as a bonus.


In [ ]:
def dedupe_ballot(ballot: List[str]) -> List[str]:
    """Keep only the first (highest-point) occurrence of each candidate on one ballot."""
    seen = set()
    cleaned = []
    for candidate in ballot:
        if candidate not in seen:
            seen.add(candidate)
            cleaned.append(candidate)
    return cleaned


messy_ballots = [["A", "A", "B"]]   # A listed twice -- should only count once, at its first (best) position
cleaned = [dedupe_ballot(b) for b in messy_ballots]
assert cleaned == [["A", "B"]]
assert tally_points(cleaned) == {"A": 3, "B": 2}
print("Ballot de-duplication works as expected.")


## Empirical complexity check

`process_ballots` should scale linearly in the total number of votes V (two O(V) passes plus an O(m log m) sort that's dominated by V for realistic candidate counts).

| Growth when n doubles | Implies |
|---|---|
| ~2x | linear |


In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark


def make_worst_case(n):
    # n ballots, each ranking 3 of 50 fixed candidates -- keeps the candidate pool
    # small and fixed so growth in n is purely "more ballots", not "more candidates".
    pool = [f"cand{i}" for i in range(50)]
    ballots = [[pool[i % 50], pool[(i + 7) % 50], pool[(i + 13) % 50]] for i in range(n)]
    return (ballots, "first")


solutions = {"process_ballots (optimal)": process_ballots}
sizes = [4000, 8000, 16000, 32000]
benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Two-pass when "first occurrence of the final state" is required.** You cannot know when something first became true if "true" is itself defined by a fact only available at the end -- tally first, then replay.
- **Collect cheap secondary facts (positional counts) in the pass you're already doing.** The positional tiebreak needed zero extra passes because it was computed alongside the point tally -- always ask "what else can I learn for free while I'm already iterating here?"
- **Tuple sort keys encode multi-level tie-breaking in one call.** Negate numeric fields you want descending; leave ascending fields (like a name) positive; always end with a field that's guaranteed unique (a name, an ID) so output is fully deterministic.
- **`3 - pos`-style formulas beat branching `if pos == 0 / elif pos == 1`.** Deriving the value from the index generalizes automatically (see the "more than 3 candidates" follow-up) and is less error-prone.
- **Related problems:** any leaderboard/ranking problem with weighted positions (Borda count voting systems), "first index where a running aggregate hits a target" problems, multi-criteria sorting (e.g. sort intervals by length then start).
- **Common pitfalls:** recording `first_reached` on a candidate's *first appearance* instead of when their running total hits their *final* score (breaks the tiebreak semantics, as the Talking Points call out with the A/B example); forgetting a final deterministic tiebreaker and getting output whose tie order depends on dict iteration order; not deduplicating a ballot that (invalidly) lists the same candidate twice.
